# Eclat

**The big picture:** the same **7,501 shopping baskets** as in Apriori. Each basket is a list of items one customer bought during a week. The goal is to find **pairs of items that are often bought together**.

This is **unsupervised** learning: there is no "right answer" column. The computer just looks for patterns.

Eclat is a simpler cousin of Apriori. Apriori scores rules with support, confidence and lift. **Eclat only cares about support**: "how often do these items appear together?" It doesn't say which item leads to the other. It just finds pairs that go together, like noticing that bread and butter keep turning up in the same baskets.

**Note:** there's no separate Eclat library here. The notebook reuses the `apriori` function from apyori and then **only looks at the support column**, which gives the ranking Eclat is after.

### Before you start
- **Read first:** **apriori**: this notebook uses the same baskets, the same preprocessing and even the same `apriori` function call, and only changes how the results are read.
- **New words in this notebook:** *eclat*, *itemset*, *support*, *frequent pair*. Look them up in the [glossary](../../../GLOSSARY.md) if you get stuck.

## Importing the libraries

- `!pip install apyori` installs a small add-on library called **apyori**, which does the rule-finding. The `!` means "run this as a command, not as Python". The output just says it was already installed. You only need to do this once per computer.
- **pandas** reads and handles tables, like Excel in Python. It reads the CSV and shows the final pairs as a neat table.
- **numpy** and **matplotlib** are imported out of habit but not really used: the result is a table, not a chart.

In [0]:
!pip install apyori

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Data Preprocessing

- `header = None` tells pandas the file has **no title row**: the very first line is already a shopping basket.
- Each of the **7,501 rows** is one customer's basket from a week at a shop, and each row lists **up to 20 items** (e.g. *burgers, meatballs, eggs*).
- The apyori library doesn't want a table. It wants a **list of baskets**, where each basket is a list of words. The loop builds exactly that:
  - `for i in range(0, 7501)` goes through every basket.
  - `for j in range(0, 20)` goes through the 20 item slots in that basket.
  - `str(...)` turns each item into text.

**A small quirk:** short baskets have empty slots. pandas fills them with a "missing" value, and `str()` turns that into the word `'nan'`. So most baskets contain a fake item called `'nan'`. It doesn't affect the results here, because the thresholds in the next step filter it out, but it is good to know.

In [0]:
dataset = pd.read_csv('Market_Basket_Optimisation.csv', header = None)
transactions = []
for i in range(0, 7501):
  transactions.append([str(dataset.values[i,j]) for j in range(0, 20)])

## Training the Eclat model on the dataset

This is the same call as in the Apriori notebook, with the same thresholds:

- `min_support = 0.003` (**support** = how often a pair appears, out of all baskets). 0.003 is 0.3% of 7,501 baskets, which is about **23 baskets**. **Why this number?** The idea is "a product bought about 3 times a day, over a 7-day week": 3 × 7 / 7,501 ≈ 0.003. Anything rarer is ignored as too rare to matter.
- `min_confidence = 0.2` and `min_lift = 3` are carried over from Apriori. Pure Eclat wouldn't use them, but leaving them in means only pairs with a real link are kept. (Without them you would get many more pairs.) In short: confidence 0.2 = "when one is bought, the other is too at least 20% of the time"; lift 3 = "3 times more likely than chance".
- `max_length = 2` caps the results at **pairs** of items. (`min_length = 2` is ignored by apyori; the confidence and lift filters are what remove single items here.)

`rules` is a "to-do" object; the work happens in the next step.

In [0]:
from apyori import apriori
rules = apriori(transactions = transactions, min_support = 0.003, min_confidence = 0.2, min_lift = 3, min_length = 2, max_length = 2)

## Visualising the results

There is no chart here. "Visualising" means turning the raw output into a readable table sorted by support.

### Displaying the first results coming directly from the output of the apriori function

- `list(rules)` makes apyori actually do the work and collect every pair that passed the thresholds.
- Typing `results` on its own prints them in raw form. Each `RelationRecord` holds `items` (the pair, e.g. `{'light cream', 'chicken'}`) and its `support`, plus extra rule details that we will ignore.

The same **9 pairs** as in Apriori come out.

In [0]:
results = list(rules)

In [0]:
results

[RelationRecord(items=frozenset({'light cream', 'chicken'}), support=0.004532728969470737, ordered_statistics=[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.29059829059829057, lift=4.84395061728395)]),
 RelationRecord(items=frozenset({'escalope', 'mushroom cream sauce'}), support=0.005732568990801226, ordered_statistics=[OrderedStatistic(items_base=frozenset({'mushroom cream sauce'}), items_add=frozenset({'escalope'}), confidence=0.3006993006993007, lift=3.790832696715049)]),
 RelationRecord(items=frozenset({'escalope', 'pasta'}), support=0.005865884548726837, ordered_statistics=[OrderedStatistic(items_base=frozenset({'pasta'}), items_add=frozenset({'escalope'}), confidence=0.3728813559322034, lift=4.700811850163794)]),
 RelationRecord(items=frozenset({'honey', 'fromage blanc'}), support=0.003332888948140248, ordered_statistics=[OrderedStatistic(items_base=frozenset({'fromage blanc'}), items_add=frozenset({'honey'}), confidence=0

### Putting the results well organised into a Pandas DataFrame

- The `inspect` function is a cut-down version of the Apriori one. It only pulls out **three** things:
  - `lhs` → **Product 1**;
  - `rhs` → **Product 2**;
  - `supports` → **Support**.
- Confidence and lift are dropped because Eclat is only about support. The columns are called "Product 1/2" rather than "Left/Right Hand Side" because a pair has no direction here.
- `zip` lines the lists up row by row, and `pd.DataFrame` turns them into a table.

In [0]:
def inspect(results):
    lhs         = [tuple(result[2][0][0])[0] for result in results]
    rhs         = [tuple(result[2][0][1])[0] for result in results]
    supports    = [result[1] for result in results]
    return list(zip(lhs, rhs, supports))
resultsinDataFrame = pd.DataFrame(inspect(results), columns = ['Product 1', 'Product 2', 'Support'])

### Displaying the results sorted by descending supports

- `nlargest(n = 10, columns = 'Support')` sorts by support, highest first, and keeps the top 10. There are only 9 pairs, so all of them are shown.

**How to read it:** support is "the share of all baskets that contain both items".
- Top pair: **herb & pepper + ground beef**, support 0.016, so about **1.6% of baskets** (roughly 120 of 7,501) contain both. That's twice as common as the next pair.
- Next: **whole wheat pasta + olive oil** (0.008, about 60 baskets), then **pasta + escalope** (0.0059).
- At the bottom, **light cream + olive oil** (0.0032) only just clears the 0.003 bar.

What "good" looks like: a high support means the pair is common, so a deal or shelf placement for it reaches many customers.

**In one sentence:** turn 7,501 shopping baskets into lists, find pairs of items bought together, and rank the pairs simply by how often they appear together (support).

In [0]:
resultsinDataFrame.nlargest(n = 10, columns = 'Support')

,Product 1,Product 2,Support
4,herb & pepper,ground beef,0.015998
7,whole wheat pasta,olive oil,0.007999
2,pasta,escalope,0.005866
1,mushroom cream sauce,escalope,0.005733
5,tomato sauce,ground beef,0.005333
8,pasta,shrimp,0.005066
0,light cream,chicken,0.004533
3,fromage blanc,honey,0.003333
6,light cream,olive oil,0.003200


## Common mistakes to avoid
- **Reading Product 1 → Product 2 as a direction.** Eclat only measures how often two items appear together, so "herb & pepper + ground beef" is the same as "ground beef + herb & pepper". The columns are just the two members of a pair.
- **Thinking this is a separate Eclat library.** It is the apyori `apriori` function with confidence and lift left out of the table. Because `min_confidence = 0.2` and `min_lift = 3` are still in the call, you get the same 9 pairs as Apriori, not every frequent pair.
- **Deleting the confidence and lift settings and expecting a longer, clean table.** apyori then also lets single items through (it ignores `min_length`), and `inspect` crashes with an IndexError because a single item has nothing on its left side. Filter to pairs first (see the second try-this idea).
- **Assuming the top pair is automatically the most useful.** High support means the pair is common, not that the link is strong. herb & pepper + ground beef has the highest support here but one of the lowest lifts in the Apriori notebook.

## Try this yourself
Change one thing, rerun the notebook, and see what happens.

1. In the training cell, change `min_support = 0.003` to `min_support = 0.006` and re-run the cells below. What to notice: only the top two pairs from the table (herb & pepper + ground beef, whole wheat pasta + olive oil) remain, because every other pair had support below 0.006.
2. In the training cell, delete `min_confidence = 0.2, min_lift = 3, ` so only the support and length settings are left, and in the cell `results = list(rules)` add a second line `results = [r for r in results if len(r.items) == 2]`. Re-run the cells below. What to notice: you get many more pairs, and the fake `'nan'` item (from empty basket slots) and very popular items like mineral water are likely to crowd the top, which is why the notebook keeps the lift filter.
3. In the last cell, change `nlargest(n = 10, columns = 'Support')` to `nlargest(n = 3, columns = 'Support')`. What to notice: you get just the three most common pairs; this is how you would pick a few pairs for a promotion.
4. In a new cell, run `print(round(0.015998 * 7501))`. What to notice: turning support back into a count shows that the top pair appears in roughly 120 baskets, which is easier to picture than a small decimal.

## Check your understanding
Try to answer in your head first, then click to reveal.

**1. What is the one score Eclat cares about, and what does it mean?**

<details><summary>Show answer</summary>

Support: the share of all baskets that contain both items. A support of 0.008 means about 0.8% of baskets (around 60 of 7,501) have both.

</details>

**2. Why does this notebook get exactly the same 9 pairs as the Apriori notebook?**

<details><summary>Show answer</summary>

Because it runs the same `apriori` call with the same thresholds, including the confidence and lift filters. Only the table changes: it keeps the support column and sorts by it.

</details>

**3. Reading the table, which pair only just made it in, and how can you tell?**

<details><summary>Show answer</summary>

light cream + olive oil, with support about 0.0032. The minimum allowed was 0.003, so it barely cleared the bar.

</details>

**4. What would happen if you removed the lift and confidence filters but changed nothing else?**

<details><summary>Show answer</summary>

apyori would also return single items, and the `inspect` function would crash with an IndexError when it tries to take the first item of an empty left side. With a pairs-only filter added, you would instead see many more pairs, including ones with the fake `'nan'` item.

</details>

---
📚 [Learning path](../../../LEARNING_PATH.md) · [Glossary](../../../GLOSSARY.md)